# Spam vs ham — text triage

Local UCI SMS data → train → **precision / recall / F1** → score one message.

*(AutoKeras search is slow — `max_trials=3` is a demo budget.)*


In [ ]:
# %pip install autokeras tensorflow scikit-learn pandas numpy

import numpy as np
import pandas as pd
import autokeras as ak
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report


In [ ]:
# Local data (vendored) — works outside Colab
path = "data/sms_spam_collection.tsv"
rows = []
with open(path, encoding="utf-8") as f:
    for line in f:
        line = line.rstrip("\n")
        if not line:
            continue
        label, text = line.split("\t", 1)
        rows.append({"label": 1 if label == "spam" else 0, "text": text})

df = pd.DataFrame(rows)
print(df["label"].value_counts())
df.head()


In [ ]:
x = np.array(df["text"].astype(str))
y = np.array(df["label"].astype(int))

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.25, random_state=222, stratify=y
)
print(len(x_train), len(x_test))


In [ ]:
# Architecture search — can take a while on CPU
clf = ak.TextClassifier(max_trials=3)
clf.fit(x_train, y_train, validation_split=0.3)


In [ ]:
y_hat = clf.predict(x_test)
print(classification_report(y_test, y_hat))


In [ ]:
text = np.array(["FW: update your account details now for your crypto wallet"])
pred = clf.predict(text)
print("SPAM" if int(pred[0]) > 0 else "not spam")
